# QanoonArena Step 3: QLoRA Supervised Fine-Tuning (A1) on Kaggle

**Group:** `G03` | **Assigned Combination:** `M2` (Fact Memory) + `A1` (QLoRA SFT)

This notebook runs **Supervised Fine-Tuning (A1)** using QLoRA on **Kaggle** with an NVIDIA GPU (**GPU T4** recommended, Internet enabled).

- All model, dataset, seed, and training hyperparameters are loaded directly from `harness/config.yaml` and preserved without modification.
- **A1 only (no DPO):** We train only the SFT LoRA adapter (`adapters/a1`).
- **Memory-agnostic adapter:** `harness.build_sft_data` formats training conversations using the baseline `M0` window so that the resulting `A1` adapter works cleanly with both `M0` and `M2` during evaluation.

## 1. Environment setup and dependency installation

Install the required QLoRA and agent dependencies while preserving Kaggle's preconfigured PyTorch/CUDA runtime and constraining Hugging Face `transformers` to `>=4.51.0,<5.0` (required for `Qwen3` support and `trl` compatibility).

In [ ]:
# Install only the required packages and pin transformers to [4.51.0, 5.0)
# Do not blindly upgrade torch or CUDA packages on Kaggle.
!pip install -q "transformers>=4.51.0,<5.0" "peft>=0.13" "trl>=0.20,<1" "accelerate>=0.30" "bitsandbytes>=0.43" "datasets>=2.19" "langgraph>=0.4,<2" "pyyaml>=6"

In [ ]:
import os

# Optional: Load HF_TOKEN from Kaggle Secrets if you need authenticated Hub downloads.
# Never hardcode tokens in notebook cells. Qwen/Qwen3-4B-Instruct-2507 is public (Apache-2.0).
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    try:
        _hf_token = _secrets.get_secret("HF_TOKEN")
        if _hf_token:
            os.environ["HF_TOKEN"] = _hf_token
            print("Loaded HF_TOKEN from Kaggle Secrets.")
    except Exception:
        print("No HF_TOKEN secret configured (optional for public Qwen/Qwen3-4B-Instruct-2507).")
except ImportError:
    print("Kaggle Secrets client unavailable (not running in Kaggle or secrets not attached).")

## 2. Kaggle GPU verification

Verify that a CUDA GPU is attached and inspect the Python, PyTorch, CUDA, Transformers, PEFT, TRL, and bitsandbytes versions.

> **Note on Kaggle `T4 x2`:** `T4 x2` provides **two separate 16 GB GPUs** (not one 32 GB GPU). If multiple GPUs are detected, this cell pins `CUDA_VISIBLE_DEVICES=0` before importing PyTorch so QLoRA runs cleanly on a single 16 GB T4 without `DataParallel` errors.

In [ ]:
import platform
import subprocess
import sys

# Inspect raw GPU hardware before importing torch
smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if smi.returncode == 0:
    print(smi.stdout)
    gpu_list = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
    n_phys_gpus = len([line for line in gpu_list.stdout.splitlines() if line.strip().startswith("GPU ")])
    if n_phys_gpus > 1 and "CUDA_VISIBLE_DEVICES" not in os.environ:
        os.environ["CUDA_VISIBLE_DEVICES"] = "0"
        print(
            f"Detected {n_phys_gpus} physical GPUs (Kaggle T4 x2 provides separate 16 GB GPUs, not 32 GB combined).\n"
            "Pinned CUDA_VISIBLE_DEVICES=0 for single-GPU QLoRA training."
        )
else:
    print("nvidia-smi failed or no NVIDIA GPU is attached.")

import torch
import transformers
import tokenizers
import peft
import trl
import accelerate
import bitsandbytes
import datasets
from packaging.version import Version

print("\n=== Environment Verification ===")
print(f"Python version      : {platform.python_version()}")
print(f"PyTorch version     : {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"Tokenizers version  : {tokenizers.__version__}")
print(f"PEFT version        : {peft.__version__}")
print(f"TRL version         : {trl.__version__}")
print(f"Accelerate version  : {accelerate.__version__}")
print(f"bitsandbytes version: {bitsandbytes.__version__}")
print(f"Datasets version    : {datasets.__version__}")
print(f"CUDA available      : {torch.cuda.is_available()}")
print(f"CUDA version        : {torch.version.cuda}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable! CPU-based QLoRA training is not supported. "
        "In Kaggle, open Settings -> Accelerator and select GPU T4 (or GPU T4 x2) with Internet enabled."
    )

tv = Version(transformers.__version__)
assert Version("4.51.0") <= tv < Version("5.0.0"), (
    f"Transformers must be >=4.51.0 and <5.0.0, found {transformers.__version__}"
)

for idx in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(idx)
    vram_gb = props.total_memory / (1024 ** 3)
    print(f"Visible GPU {idx}       : {props.name} ({vram_gb:.2f} GB VRAM)")

## 3. Repository setup and working directory

Locate the `qanon_arena` repository (or clone it into `/kaggle/working/qanon_arena/` if not already present), set the working directory and Python import path, and configure Kaggle output paths under `/kaggle/working/`.

In [ ]:
import shutil
from pathlib import Path

# ---- Group G03 / A1 SFT settings -----------------------------------------
REPO_URL   = "https://github.com/mubarizkyc/qanon_arena.git"
GROUP      = "G03"
TRAIN_TYPE = "A1"          # G03 is assigned A1 (QLoRA SFT only; no DPO)
# --------------------------------------------------------------------------

assert GROUP == "G03", f"Expected GROUP='G03', got {GROUP!r}"
assert TRAIN_TYPE == "A1", f"Expected TRAIN_TYPE='A1' (SFT only), got {TRAIN_TYPE!r}"

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
INPUT_DIR = Path("/kaggle/input") if Path("/kaggle/input").exists() else None
EXPECTED_REPO_DIR = WORK_DIR / "qanon_arena"

def is_repo_root(p: Path) -> bool:
    return (p / "harness" / "config.yaml").is_file() and (p / "harness" / "build_sft_data.py").is_file()

if is_repo_root(Path.cwd()):
    REPO_DIR = Path.cwd()
elif is_repo_root(EXPECTED_REPO_DIR):
    REPO_DIR = EXPECTED_REPO_DIR
else:
    input_repo = None
    if INPUT_DIR is not None:
        for cand in INPUT_DIR.glob("*"):
            if is_repo_root(cand):
                input_repo = cand
                break
            nested = cand / "qanon_arena"
            if is_repo_root(nested):
                input_repo = nested
                break
    if input_repo is not None:
        print(f"Copying repository from Kaggle input {input_repo} -> {EXPECTED_REPO_DIR}")
        shutil.copytree(input_repo, EXPECTED_REPO_DIR, dirs_exist_ok=True)
    else:
        print(f"Cloning {REPO_URL} -> {EXPECTED_REPO_DIR}")
        subprocess.run(["git", "clone", REPO_URL, str(EXPECTED_REPO_DIR)], check=True)
    REPO_DIR = EXPECTED_REPO_DIR

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

ADAPTER_DIR = WORK_DIR / "adapters" / "a1"
SFT_OUTPUT_DIR = WORK_DIR / "outputs" / "sft"
LOSS_PLOT_PATH = WORK_DIR / "sft_loss.png"
ZIP_PATH = WORK_DIR / "a1_adapter.zip"

print("Working directory :", os.getcwd())
print("Output root       :", WORK_DIR)
print("Adapter directory :", ADAPTER_DIR)
print("ZIP output path   :", ZIP_PATH)

## 4. Dataset and model verification

1. Load the fixed training settings and pinned base model (`Qwen/Qwen3-4B-Instruct-2507`) from `harness/config.yaml`.
2. Build `data_sft.jsonl` using `harness.build_sft_data` (preserving the exact chatbot prompt format and `M0` window so the `A1` adapter stays compatible with both `M0` and `M2`).
3. Run `harness.leakage_check` to verify zero overlap with `dev`, `smoke`, `secret test`, or contributed questions.
4. Verify dataset schema, non-emptiness, and base model accessibility.

In [ ]:
import json
import random
import yaml
from transformers import set_seed

with open("harness/config.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

T = cfg["training"]
BASE = cfg["model"]["base_model"]
SEED = cfg["seeds"][0]

random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

print("Base model      :", BASE)
print("Load in 4-bit   :", cfg["model"]["load_in_4bit"])
print("Seed            :", SEED)
print("Training budget :", T)

# Verify required instructor-provided data and corpus files exist
for req_path in [
    Path("harness/data/train.jsonl"),
    Path("harness/corpus/statutes.jsonl"),
    Path("harness/corpus/judgments.jsonl"),
]:
    if not req_path.is_file() or req_path.stat().st_size == 0:
        raise FileNotFoundError(f"Required instructor resource is missing or empty: {req_path}")

# Build SFT dataset and fail immediately if leakage_check reports any overlap
SFT_DATA_PATH = REPO_DIR / "data_sft.jsonl"
subprocess.run([sys.executable, "-m", "harness.build_sft_data", "--out", str(SFT_DATA_PATH)], check=True)
subprocess.run([sys.executable, "-m", "harness.leakage_check", str(SFT_DATA_PATH)], check=True)

In [ ]:
from datasets import Dataset
from transformers import AutoConfig, AutoTokenizer

# 1 & 2. Verify dataset exists and is nonempty
assert SFT_DATA_PATH.is_file(), f"Missing generated SFT dataset: {SFT_DATA_PATH}"
rows = [json.loads(line) for line in SFT_DATA_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
assert len(rows) > 0, "SFT dataset is empty!"
assert len(rows) <= T["max_train_examples"], (
    f"Example count {len(rows)} exceeds max_train_examples={T['max_train_examples']}"
)

# 3. Verify required JSONL fields and target JSON format
for i, r in enumerate(rows):
    assert isinstance(r.get("messages"), list) and len(r["messages"]) >= 2, f"Row {i} missing valid 'messages'"
    for m in r["messages"]:
        assert "role" in m and "content" in m and isinstance(m["content"], str), f"Row {i} malformed message"
    assert r["messages"][-1]["role"] == "assistant", f"Row {i} last message must be assistant"
    parsed = json.loads(r["messages"][-1]["content"])
    assert {"answer", "citations", "abstain"}.issubset(parsed.keys()), (
        f"Row {i} assistant completion missing required answer keys"
    )

# Prompt-completion format: loss is computed only on the assistant completion
sft_ds = Dataset.from_list([
    {"prompt": r["messages"][:-1], "completion": [r["messages"][-1]]}
    for r in rows
])
sft_ds = sft_ds.shuffle(seed=SEED)

print(f"Verified {len(sft_ds)} SFT training examples (limit {T['max_train_examples']}).")
print("Dataset columns        :", sft_ds.column_names)
print("Sample assistant target:", rows[0]["messages"][-1]["content"])

# 4. Verify the pinned base model is accessible (via Hub or optional attached Kaggle Input)
MODEL_SOURCE = BASE
if INPUT_DIR is not None:
    model_slug = BASE.split("/")[-1].lower()
    for cand in INPUT_DIR.rglob("config.json"):
        if model_slug in str(cand.parent).lower() and (cand.parent / "tokenizer_config.json").is_file():
            MODEL_SOURCE = str(cand.parent)
            print(f"Found local Kaggle input model matching {BASE}: {MODEL_SOURCE}")
            break

try:
    model_cfg = AutoConfig.from_pretrained(MODEL_SOURCE)
    tok_check = AutoTokenizer.from_pretrained(MODEL_SOURCE)
    print(f"Verified base model '{BASE}' (arch={model_cfg.architectures}, vocab_size={len(tok_check)}).")
    del tok_check
except Exception as exc:
    raise RuntimeError(
        f"Unable to access base model '{MODEL_SOURCE}'. "
        "Verify Kaggle Internet is enabled or attach the model via Kaggle Input / HF_TOKEN secret."
    ) from exc

## 5. QLoRA configuration

Load `Qwen/Qwen3-4B-Instruct-2507` in 4-bit NF4 quantization (`float16` compute dtype for T4), configure the LoRA adapter using the exact parameters from `harness/config.yaml`, and verify that all target modules exist in the loaded model architecture.

In [ ]:
import inspect
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig

tok = AutoTokenizer.from_pretrained(MODEL_SOURCE)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

def load_base():
    m = AutoModelForCausalLM.from_pretrained(
        MODEL_SOURCE,
        quantization_config=bnb,
        device_map={"": 0},
        torch_dtype=torch.float16,
    )
    m.config.use_cache = False
    return m

TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
lora = LoraConfig(
    r=T["lora_r"],
    lora_alpha=T["lora_alpha"],
    lora_dropout=T["lora_dropout"],
    target_modules=TARGET_MODULES,
    task_type="CAUSAL_LM",
)

def make_args(cls, **kw):
    """Pass only the settings this TRL version knows (names changed between versions)."""
    names = set(getattr(cls, "__dataclass_fields__", {})) or set(inspect.signature(cls).parameters)
    if "max_length" not in names and "max_seq_length" in names and "max_length" in kw:
        kw["max_seq_length"] = kw.pop("max_length")
    dropped = [k for k in kw if k not in names]
    if dropped:
        print("note: this TRL version ignores", dropped)
    return cls(**{k: v for k, v in kw.items() if k in names})

def make_trainer(cls, **kw):
    params = inspect.signature(cls.__init__).parameters
    if "processing_class" not in params and "processing_class" in kw:
        kw["tokenizer"] = kw.pop("processing_class")
    return cls(**kw)

model = load_base()
assert getattr(model, "is_loaded_in_4bit", False), "Expected model to be loaded in 4-bit quantization!"
leaf_module_names = {name.split(".")[-1] for name, _ in model.named_modules()}
missing_targets = [m for m in TARGET_MODULES if m not in leaf_module_names]
assert not missing_targets, f"LoRA target modules missing from {BASE}: {missing_targets}"

print("Base model loaded in 4-bit on", next(model.parameters()).device)
print("Verified LoRA target modules:", TARGET_MODULES)
print("LoRA config:", {"r": lora.r, "lora_alpha": lora.lora_alpha, "lora_dropout": lora.lora_dropout})

## 6. SFT training

Train the `A1` adapter with `SFTTrainer` using the exact instructor-approved hyperparameters (`epochs=2`, `learning_rate=2e-4`, `max_seq_len=2048`, `seed=13`, `per_device_train_batch_size=1`, `gradient_accumulation_steps=8`, `fp16=True`, `completion_only_loss=True`) and record wall-clock training time.

In [ ]:
import time
from datetime import datetime, timezone
from trl import SFTConfig, SFTTrainer

torch.cuda.reset_peak_memory_stats()

sft_args = make_args(
    SFTConfig,
    output_dir=str(SFT_OUTPUT_DIR),
    num_train_epochs=T["epochs"],
    learning_rate=T["learning_rate"],
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="no",
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    max_length=T["max_seq_len"],
    completion_only_loss=True,
    seed=SEED,
    report_to="none",
)

trainer = make_trainer(
    SFTTrainer,
    model=model,
    args=sft_args,
    train_dataset=sft_ds,
    processing_class=tok,
    peft_config=lora,
)

# Verify QLoRA adapter-only training (not full-parameter fine-tuning)
trainable_params = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in trainer.model.parameters())
assert trainable_params > 0, "No trainable parameters found!"
assert trainable_params < total_params * 0.1, (
    f"Trainable parameter ratio too high ({trainable_params}/{total_params}); expected QLoRA adapter only."
)
assert len(trainer.train_dataset) == len(sft_ds), "Trainer dataset size mismatch!"
trainer.model.print_trainable_parameters()

train_start_dt = datetime.now(timezone.utc)
t0 = time.perf_counter()
train_result = trainer.train()
t1 = time.perf_counter()
train_end_dt = datetime.now(timezone.utc)

sft_seconds = t1 - t0
sft_minutes = sft_seconds / 60.0
sft_hours = sft_seconds / 3600.0
peak_vram_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)

print(f"SFT done in {sft_minutes:.2f} min ({sft_seconds:.1f} s | {sft_hours:.4f} GPU-hours)")

## 7. Training metrics and timing

Plot the actual trainer loss history to `/kaggle/working/sft_loss.png` and print the exact timing, GPU memory, and loss metrics needed for `MODEL_CARD.md` and `report/section.tex`.

In [ ]:
import matplotlib.pyplot as plt

hist = [h for h in trainer.state.log_history if "loss" in h]
steps = [h["step"] for h in hist]
losses = [h["loss"] for h in hist]

if hist:
    plt.figure(figsize=(8, 4.5))
    plt.plot(steps, losses, marker="o", markersize=3, linewidth=1.5)
    plt.xlabel("step")
    plt.ylabel("training loss")
    plt.title(f"QanoonArena {GROUP} A1 (QLoRA SFT) Training Loss")
    plt.grid(True, alpha=0.3)
    plt.savefig(LOSS_PLOT_PATH, dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Warning: no training loss entries recorded in trainer.state.log_history.")

gpu_name = torch.cuda.get_device_name(0)
gpu_total_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
avg_train_loss = getattr(train_result, "training_loss", None)

training_report = {
    "group": GROUP,
    "assigned_memory": "M2 (Fact Memory; A1 trained memory-agnostic)",
    "adapter": TRAIN_TYPE,
    "base_model": BASE,
    "train_examples": len(sft_ds),
    "training_config": {
        "epochs": T["epochs"],
        "lora_r": T["lora_r"],
        "lora_alpha": T["lora_alpha"],
        "lora_dropout": T["lora_dropout"],
        "learning_rate": T["learning_rate"],
        "max_seq_len": T["max_seq_len"],
        "per_device_train_batch_size": 1,
        "gradient_accumulation_steps": 8,
        "lr_scheduler_type": "cosine",
        "warmup_ratio": 0.03,
        "seed": SEED,
    },
    "gpu": {
        "model": gpu_name,
        "total_vram_gb": round(gpu_total_gb, 2),
        "peak_allocated_vram_gb": round(peak_vram_gb, 2),
    },
    "timing": {
        "start_utc": train_start_dt.isoformat(),
        "end_utc": train_end_dt.isoformat(),
        "total_seconds": round(sft_seconds, 2),
        "total_minutes": round(sft_minutes, 2),
        "gpu_hours": round(sft_hours, 4),
    },
    "loss": {
        "initial_logged_loss": losses[0] if losses else None,
        "final_logged_loss": losses[-1] if losses else None,
        "average_training_loss": round(avg_train_loss, 6) if avg_train_loss is not None else None,
        "logged_steps": len(hist),
    },
    "loss_plot": str(LOSS_PLOT_PATH),
}

print("\n=== Training Summary (for MODEL_CARD.md and report/section.tex) ===")
print(json.dumps(training_report, indent=2))

## 8. Adapter saving and validation

Save the unmerged PEFT adapter and tokenizer to `/kaggle/working/adapters/a1/`, verify all required files and metadata, confirm the adapter is not merged into the base model, and reload it onto a fresh 4-bit base model.

In [ ]:
import gc
from peft import PeftModel

ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
trainer.model.save_pretrained(str(ADAPTER_DIR))
tok.save_pretrained(str(ADAPTER_DIR))
print(f"Saved PEFT adapter and tokenizer to: {ADAPTER_DIR}")

# 1. Directory exists
assert ADAPTER_DIR.is_dir(), f"Adapter directory missing: {ADAPTER_DIR}"

# 2. adapter_config.json exists
cfg_file = ADAPTER_DIR / "adapter_config.json"
assert cfg_file.is_file(), f"Missing adapter_config.json in {ADAPTER_DIR}"
saved_adapter_cfg = json.loads(cfg_file.read_text(encoding="utf-8"))

# 3. Adapter weights exist
has_safetensors = (ADAPTER_DIR / "adapter_model.safetensors").is_file()
has_bin = (ADAPTER_DIR / "adapter_model.bin").is_file()
assert has_safetensors or has_bin, f"Missing adapter weight file in {ADAPTER_DIR}"

# 4. Verify saved config points to the official base model (and normalize if loaded from local Kaggle Input)
saved_base = saved_adapter_cfg.get("base_model_name_or_path")
assert saved_base in (BASE, MODEL_SOURCE), f"Unexpected base_model_name_or_path: {saved_base}"
if saved_base != BASE:
    saved_adapter_cfg["base_model_name_or_path"] = BASE
    cfg_file.write_text(json.dumps(saved_adapter_cfg, indent=2), encoding="utf-8")
    print(f"Normalized base_model_name_or_path -> {BASE}")

assert saved_adapter_cfg.get("peft_type") == "LORA", f"Unexpected peft_type: {saved_adapter_cfg.get('peft_type')}"
assert saved_adapter_cfg.get("r") == T["lora_r"], f"Rank mismatch: {saved_adapter_cfg.get('r')}"
assert saved_adapter_cfg.get("lora_alpha") == T["lora_alpha"], f"Alpha mismatch: {saved_adapter_cfg.get('lora_alpha')}"

# 6. Verify adapter was NOT merged into a full model checkpoint
merged_weights = [
    p.name for p in ADAPTER_DIR.iterdir()
    if p.name.startswith("model") and p.suffix in (".safetensors", ".bin")
]
assert not merged_weights, f"Unexpected full-model weight files found (adapter appears merged): {merged_weights}"

# Mirror to <repo>/adapters/a1 so relative `--adapter-path adapters/a1` works in the same session
REPO_ADAPTER_DIR = REPO_DIR / "adapters" / "a1"
if REPO_ADAPTER_DIR.resolve() != ADAPTER_DIR.resolve():
    shutil.copytree(ADAPTER_DIR, REPO_ADAPTER_DIR, dirs_exist_ok=True)
    print(f"Mirrored adapter to repo path   : {REPO_ADAPTER_DIR}")

print("Saved adapter files:")
for p in sorted(ADAPTER_DIR.iterdir()):
    print(f"  - {p.name} ({p.stat().st_size / 1024:.1f} KB)")

# 5. Free training state and verify reloading with the base model + PEFT
del trainer, model
gc.collect()
torch.cuda.empty_cache()

base_check = load_base()
reloaded_model = PeftModel.from_pretrained(base_check, str(ADAPTER_DIR))
reloaded_model.eval()
assert isinstance(reloaded_model, PeftModel) and hasattr(reloaded_model, "peft_config")
print("Reload validation PASSED: saved A1 adapter loads cleanly onto fresh 4-bit base model.")

del reloaded_model, base_check
gc.collect()
torch.cuda.empty_cache()

## 9. Adapter ZIP creation

Package `/kaggle/working/adapters/a1/` into `/kaggle/working/a1_adapter.zip` so that all entries sit inside `a1/` (extract into your local repository's `adapters/` folder to get `adapters/a1/`).

In [ ]:
import zipfile

created_zip = Path(shutil.make_archive(
    str(WORK_DIR / "a1_adapter"),
    "zip",
    root_dir=str(ADAPTER_DIR.parent),
    base_dir=ADAPTER_DIR.name,
))
assert created_zip.is_file() and created_zip.stat().st_size > 0, "ZIP creation failed!"

with zipfile.ZipFile(created_zip, "r") as zf:
    names = zf.namelist()
    assert "a1/adapter_config.json" in names, f"Unexpected ZIP structure: {names[:10]}"
    print(f"Created {created_zip} ({created_zip.stat().st_size / (1024 * 1024):.2f} MB) with {len(names)} entries:")
    for info in zf.infolist():
        print(f"  - {info.filename} ({info.file_size / 1024:.1f} KB)")

## 10. Final checklist and next steps

### After this notebook finishes on Kaggle
1. Download `/kaggle/working/a1_adapter.zip` and `/kaggle/working/sft_loss.png` from the Kaggle **Output** panel.
2. Extract `a1_adapter.zip` into your local `qanon_arena/adapters/` directory so the path is `adapters/a1/` (containing `adapter_config.json`, `adapter_model.safetensors`, and tokenizer files).
3. Update `MODEL_CARD.md` (using the printed training summary and `sft_loss.png`) and `report/section.tex`.

### Four-version evaluation commands (Group `G03`: `M2` + `A1`, all 3 seeds)
Run these from the repository root (`qanon_arena/`) once `agent/memory/m2_facts.py` and `adapters/a1/` are ready:

```bash
python -m harness.evaluate --group G03 --memory M0 --adapter A0 --split dev --all-seeds
python -m harness.evaluate --group G03 --memory M2 --adapter A0 --split dev --all-seeds
python -m harness.evaluate --group G03 --memory M0 --adapter A1 --adapter-path adapters/a1 --split dev --all-seeds
python -m harness.evaluate --group G03 --memory M2 --adapter A1 --adapter-path adapters/a1 --split dev --all-seeds
python -m harness.summarize --group G03 --split dev
```